NOMES E RMS:

Enzo Coppa Selingarde RM 573393
Gabriel Carlos Barbosa RM 574074
Gustavo de Souza Abreu RM 574080

In [1]:
!pip install -q huggingface_hub

In [12]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HUGGING_FACE_API_KEY")
login(token=HF_TOKEN)

In [16]:
modelo = "meta-llama/Llama-3.2-3B-Instruct"

import random

from transformers import pipeline
import torch

print("Iniciando o download/carregamento do modelo localmente...")

gerador_local = pipeline(
    "text-generation",
    model=modelo,
    torch_dtype=torch.float16, 
    device_map="auto" 
)

Iniciando o download/carregamento do modelo localmente...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [4]:
def classificar_temperatura(temp):
    if 18 <= temp <= 30:
        return "Seguro"
    elif 30 < temp <= 38:
        return "Alerta"
    else:
        return "Crítico"

def classificar_luminosidade(lumi):
    if 0 <= lumi <= 341:
        return "Seguro"
    elif 342 <= lumi <= 682:
        return "Alerta"
    else:
        return "Crítico"

def classificar_vibracao(vibra):
    if vibra == "Sim":
        return "Crítico"
    else:
        return "Seguro"

def classificar_bateria(bateria):
    if 51 <= bateria <= 100:
        return "Seguro"
    elif 30 <= bateria <= 50:
        return "Alerta"
    else:
        return "Crítico"

def classificar_solar(geracaosol):
    if 667 <= geracaosol <= 1000:
        return "Seguro"
    elif 334 <= geracaosol <= 666:
        return "Alerta"
    else:
        return "Crítico"
        
def classificar_comunicacao(comunicacao):
    if 67 <= comunicacao <= 100:
        return "Seguro"
    elif 34 <= comunicacao <= 66:
        return "Alerta"
    else:
        return "Crítico"

In [5]:
temperatura = random.uniform(15.0, 50.0)
lumi = random.randint(0, 1023)
vibra = random.choice(["Sim","Não"])
bateria = random.randint(0, 100)
geracaosol = random.randint(0, 1000)
comunicacao = random.randint(0, 100)

status_temp = classificar_temperatura(temperatura)
status_lumi = classificar_luminosidade(lumi)
status_vibra = classificar_vibracao(vibra)
status_bateria = classificar_bateria(bateria)
status_solar = classificar_solar(geracaosol)
status_comunicacao = classificar_comunicacao(comunicacao)

print(f"Temperatura: {temperatura:.1f}°C - {status_temp} | Luminosidade: {lumi} - {status_lumi} | Vibração: {vibra} - {status_vibra} | Bateria: {bateria}% - {status_bateria}  | Exposição Solar {geracaosol}W - {status_solar}  | Comunicação {comunicacao}% - {status_comunicacao} |")

Temperatura: 45.7°C - Crítico | Luminosidade: 457 - Alerta | Vibração: Sim - Crítico | Bateria: 5% - Crítico  | Exposição Solar 711W - Seguro  | Comunicação 31% - Crítico |


In [6]:
mensagens = [
    
    {"role": "system", "content": """Você é um agente especialista em astronáutica. Você recebe leituras de sensores de uma cabine espacial: temperatura, luminosidade e vibração. Sua função é prever falhas e recomendar ações com base na seguinte tabela de risco:
Temperatura: seguro 18–30°C, alerta 30–38°C, crítico >38°C
Luminosidade: seguro 0–341, alerta 342–682, crítico >683
Vibração: crítico se detectada

Bateria: seguro 51-100%, alerta 30-50%, crítico 0-29%
Geração Solar: seguro 667-1000W, alerta 334-666W, crítico 0-333W
Comunicação: seguro 67-100%, alerta 34-66%, crítico 0-33%

Para classificar, siga EXATAMENTE: se temperatura entre 30-38°C = ALERTA, não crítico. Só é crítico se acima de 38°C.

No campo Risco, escreva apenas as consequências físicas para a tripulação e a nave. Por exemplo: superaquecimento pode danificar equipamentos, exposição à radiação pode prejudicar a tripulação.
Não omita nenhum campo. O campo Recomendação é obrigatório.
Responda sempre em português, de forma clara e estruturada,IMPORTANTE: Responda SEMPRE em português brasileiro. Nunca responda em inglês, e Antes de responder, classifique cada sensor individualmente comparando o valor recebido com a tabela de risco. Nunca repita os dados no campo Risco — explique a consequência do valor medido. por exemplo:
🔴 Status: Crítico
⚠️ Risco: ...
✅ Recomendação: ..."""},


  {"role": "user", "content": f"""Leitura atual dos sensores:
- Temperatura: {temperatura:.1f}°C — Status: {status_temp}
- Luminosidade: {lumi} — Status: {status_lumi}
- Vibração: {vibra} — Status: {status_vibra}
- Bateria {bateria}% — Status: {status_bateria}
- Exposição Solar {geracaosol}W — Status: {status_solar}
- Comunicação {comunicacao}% — Status: {status_comunicacao}

Analise cada sensor e responda no formato:
🔴 Status geral: [nível]
⚠️ Risco: [consequência de cada sensor]
✅ Recomendação: [ação concreta]"""}

]



In [7]:
prompt = gerador_local.tokenizer.apply_chat_template(
    mensagens,
    tokenize=False,
    add_generation_prompt=True
)

outputs = gerador_local(
    prompt,
    max_new_tokens=1000,
    do_sample=True,
    temperature=0.4,
)

resposta = outputs[0]["generated_text"][len(prompt):]
print(resposta)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Aqui estão as análises e recomendações para cada sensor:

- 🔴 Status geral: CRÍTICO
⚠️ Risco: Temperatura - Superaquecimento pode danificar equipamentos, exposição prolongada pode causar danos à tripulação.
✅ Recomendação: Verificar e ajustar o sistema de refrigeração o mais rápido possível.

- 🔴 Status geral: ALERTA
⚠️ Risco: Luminosidade - Exposição prolongada a níveis altos de radiação pode prejudicar a visão e a saúde da tripulação.
✅ Recomendação: Ajustar a orientação da cabine para minimizar a exposição à radiação solar.

- 🔴 Status geral: CRÍTICO
⚠️ Risco: Vibração - Pode causar danos à estrutura da cabine e aumentar o risco de acidentes.
✅ Recomendação: Verificar e ajustar o sistema de estabilização da cabine.

- 🔴 Status geral: CRÍTICO
⚠️ Risco: Bateria - Falta de energia pode causar perda de comunicação e sistemas críticos.
✅ Recomendação: Trocar a bateria o mais rápido possível.

- 🔴 Status geral: CRÍTICO
⚠️ Risco: Comunicação - Falta de comunicação pode aumentar o risco de 

In [8]:
print("\n--- Modo de perguntas ---")
print("Você pode fazer perguntas sobre as leituras. Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Sua pergunta: ")
    
    if pergunta.lower() == "sair":
        print("Encerrando sistema de monitoramento.")
        break
    
    mensagens_chat = [
        {"role": "system", "content": """Você é um agente especialista em astronáutica. Você recebe leituras de sensores de uma cabine espacial: temperatura, luminosidade e vibração. Sua função é prever falhas e recomendar ações com base na seguinte tabela de risco:
Temperatura: seguro 18–30°C, alerta 30–38°C, crítico >38°C
Luminosidade: seguro 0–341, alerta 342–682, crítico >683
Vibração: crítico se detectada

Bateria: seguro 51-100%, alerta 30-50%, crítico 0-29%
Geração Solar: seguro 667-1000W, alerta 334-666W, crítico 0-333W
Comunicação: seguro 67-100%, alerta 34-66%, crítico 0-33%

Responda SEMPRE em português brasileiro. Nunca responda em inglês."""},
        {"role": "user", "content": f"""Leituras atuais: Temperatura: {temperatura:.1f}°C ({status_temp}), Luminosidade: {lumi} ({status_lumi}), Vibração: {vibra} ({status_vibra}). Bateria: ({bateria}%) ({status_bateria}),
 Exposição Solar: ({geracaosol}W) ({status_solar}), Comunicação: ({comunicacao}%) ({status_comunicacao}),Pergunta: {pergunta}"""}

    ]
    
    prompt_chat = gerador_local.tokenizer.apply_chat_template(
        mensagens_chat,
        tokenize=False,
        add_generation_prompt=True
    )
    
    resposta_chat = gerador_local(
        prompt_chat,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.4,
    )
    
    print("\nResposta:", resposta_chat[0]["generated_text"][len(prompt_chat):])
    print()


--- Modo de perguntas ---
Você pode fazer perguntas sobre as leituras. Digite 'sair' para encerrar.



Sua pergunta:  Quais sistemas precisam de atenção imediata?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Com base nas leituras atuais, há três sistemas que precisam de atenção imediata:

1. **Bateria**: Está em 5% e está em nível crítico. É fundamental realizar medidas para aumentar a capacidade de energia disponível, pois uma falha pode ocorrer a qualquer momento.

2. **Comunicação**: Está em 31% e está em nível crítico. Uma falha na comunicação pode ser perigosa, pois afeta a capacidade de receber e enviar informações essenciais para o controle e a segurança da missão.

3. **Vibração**: Está detectada e está em nível crítico. Uma vibração crítica pode indicar problemas mecânicos graves, como uma falha no sistema de sustentação ou no motor, que precisam ser investigados e corrigidos imediatamente.

É fundamental priorizar a atenção



KeyboardInterrupt: Interrupted by user